# Notebook 1 — Fire & Smoke Detection (YOLOv8n)
**Kaggle T4x2 · ~30 phút**

In [ ]:
!nvidia-smi
import torch
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
from pathlib import Path

# ── Đường dẫn gốc ─────────────────────────────────────────────────────────────
BASE      = Path('/kaggle/input/datasets/nguynteincong/dataset-fall-fire/datasets')
FIRE_ROOT = BASE / 'Fire_and_Smoke'
# Kiểm tra
for split in ['train', 'valid', 'test']:
    imgs = len(list((FIRE_ROOT / split / 'images').glob('*')))
    lbs  = len(list((FIRE_ROOT / split / 'labels').glob('*')))
    print(f'{split:6s} → images: {imgs:4d}  labels: {lbs:4d}')

In [ ]:
# Tạo data.yaml với path đúng
yaml_content = f"""path: {FIRE_ROOT}
train: train/images
val:   valid/images
test:  test/images

nc: 2
names: ['fire', 'smoke']
"""

yaml_path = '/kaggle/working/fire_data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data     = yaml_path,
    epochs   = 100,
    imgsz    = 416,
    batch    = 32,
    device   = '0,1',
    cache    = True,
    project  = '/kaggle/working/runs',
    name     = 'fire_v1',
    patience = 20,
    save     = True,
    plots    = True,
)

In [ ]:
import glob

best_path  = glob.glob('/kaggle/working/runs/fire_v1/weights/best.pt')[0]
best_model = YOLO(best_path)

metrics = best_model.val(data=yaml_path, split='test', imgsz=416, device=0)
print(f'mAP50    : {metrics.box.map50:.4f}  (mục tiêu > 0.85)')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall   : {metrics.box.mr:.4f}')

In [ ]:
best_model.export(format='onnx', imgsz=416, opset=12, simplify=True)

import shutil
onnx_src = best_path.replace('.pt', '.onnx')
shutil.copy(best_path, '/kaggle/working/fire_best.pt')
shutil.copy(onnx_src,  '/kaggle/working/fire_best.onnx')

print('Done:')
print('  /kaggle/working/fire_best.pt')
print('  /kaggle/working/fire_best.onnx')